# 7 (v2). External training, climate matching and transfer to Atok

**What this notebook is.** An exploratory feasibility study of transferring crop classifiers trained on external datasets (CropHarvest, Sen4AgriNet, FranceCrops) to Atok. **It is not a validated crop-classification prototype.** The Atok labels are advisor-described rotations without recorded planting dates (the Paoay KML), repeated across 2019-2025, and only 2 carrot, 2 cabbage and 1 potato field are labelled. Every Atok score measures agreement with that assumed calendar on very few independent fields. Section 9 sets out the acceptance criteria a verified pilot would have to meet.

This revision responds to a methodological review of the first version. Its main additions:

- **Baselines:** majority-class and random baselines, plus balanced accuracy, next to every score.
- **Model comparison:** RF and LR compared on the same predictors.
- **Overfitting checks:** a grouped-CV tuning grid, per-crop learning curves at the prototype's 1,000 trees, grouped permutation importance and feature-family ablation.
- **Calibration:** reliability measured, then a calibration fitted on grouped out-of-fold predictions; the abstention threshold is tested on unseen regions; out-of-distribution check; risk-coverage curves and rejection causes.
- **Transfer tests:** leave-one-region-out runs and a train-region × test-region matrix.
- **Controlled climate test:** primary test on external target regions with verified labels (every eligible source set per target, per-target robustness). Atok is an exploratory application only.
- **Comparable transfer scores:** split by number of shared crops, plus one-vs-rest AUC per crop.
- **Local benchmark recast** as a weak-label fitting diagnostic; field-independent validation reported as unavailable.
- **Domain shift:** temporal signatures compared by crop and source.
- **Maps:** an unknown class, the best- and worst-coverage years, and disagreement maps with pixel counts.

| Section | Objective | Output |
|---|---|---|
| 1. Data inventory and profiling | Obj 1 | Inventory, independent sample units, region × crop coverage, calendar coverage, climate distances, domain shift |
| 2. Preprocessing QC | Obj 2 | Pointer to notebooks 01-05 and their audits |
| 3. Dataset specification | Obj 3a | Spec table, unit-by-unit exclusion funnel, record-level CropHarvest exclusions |
| 4. External models and diagnostics | Obj 3b | Same-predictor comparison, tuning, learning curves, importance, calibration, leave-one-region-out |
| 5. Climate-matching experiment | Obj 3b | All eligible region sets: climate distance vs Atok agreement, size-matched control |
| 6. Atok evaluation (provisional labels) | - | Baselines, sample units, clear-data sensitivity, domain shift, failure analysis, exploratory local benchmark |
| 7. Atok maps (exploratory) | Obj 3b | Class maps with an unknown class, two years, disagreement maps |
| 8-9. Limitations, requirements and acceptance criteria | Obj 4 | Evidence-backed limitations, proposed pilot acceptance criteria and current status |

Outputs go to `analysis/transferability/`. This notebook reuses v1's cached tables, locked test split and CV results from `analysis/external/`. Inputs are set in `settings.local.json` (see settings.example.json).

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from crop_pipeline.transferability import atok as atok_data
from crop_pipeline.transferability import climate, config, experiments, external, maps, models, plots, report
from crop_pipeline.transferability.config import CLASSES, INTERVAL, LR_SHAPE, MIN_CLEAR, OOD_QUANTILE, PROTOTYPE

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
warnings.filterwarnings("ignore", message="Geometry is in a geographic CRS")
pd.set_option("display.precision", 3)
pd.set_option("display.width", 220)

In [ ]:
config.chdir_to_repository()
paths = config.load_paths()  # settings.local.json; keys are listed in settings.example.json
config.check_inputs(paths)
OUTPUT, FIGURES = paths.output, paths.figures
print("Output folder:", OUTPUT)

## Inputs

**External tables.** These are v1's cached CropHarvest and Sen4AgriNet/FranceCrops tables, with the same harmonisation, SAVI and shape features, 10 m parcel linking and locked test split. The split is checked against v1's `test_split.csv` and must match exactly.

**Atok.** This uses the ROI cube cached by notebook 08 or 09 (half-month Cloud Score+ composites, 2019-2026) and the KML polygons. An Atok sample is one pixel in one calendar window: 8 half-months starting at the calendar planting month. It needs at least 6 clear composites; gaps are interpolated linearly.

In [ ]:
tables, data = external.load_data(paths)  # v1's cached tables and locked test split
development, test = external.split_partitions(data)
print(f"External rows: {len(data):,}; parcels: {data['partition_group'].nunique()}; "
      f"test parcels: {test['partition_group'].nunique()}")

In [ ]:
roi = atok_data.load_grid(paths)  # ROI cube cached by notebook 08/09, with the KML fields on its grid
print(f"Atok cube {roi.cube.shape}, {roi.composites} composites; interior pixels:",
      {name: int(mask.sum()) for name, mask in roi.interiors.items()})

In [ ]:
atok_all, atok, atok_windows = atok_data.labelled_samples(roi)  # provisional labels: atok_data.LABEL_ROWS
print(f"Atok labelled pixel-windows (>= {MIN_CLEAR} clear): {len(atok):,} of {atok_windows['pixel_windows'].sum():,}")

In [ ]:
atok_units = atok_data.sample_units(atok)
atok_units.to_csv(OUTPUT / "atok_sample_units.csv")
print("Primary sample sizes are the fields and field-seasons; pixel-windows within a field are not independent:")
display(atok_units)

## 1. Data inventory and profiling (Obj 1)

### 1.1 Inventory

There is one row per source and crop. For the external sources, `samples` = `pixels` (one window per pixel); for Atok, `pixels` counts unique pixels and `samples` counts pixel-windows across 2019-2025. `parcels` means parcels for the external sources and KML fields for Atok. Label provenance separates verified labels from published datasets from the advisor-described rotation. B2 (lettuce) and Area 5 (seedbeds) are listed, but the model has no `other_crop` class to use them.

In [ ]:
inventory = external.inventory(data, atok, roi.interiors)
inventory.to_csv(OUTPUT / "inventory.csv", index=False)
display(inventory)

In [ ]:
imbalance = external.imbalance(inventory)
imbalance.to_csv(OUTPUT / "imbalance.csv", index=False)
print("Imbalance ratios (largest ÷ smallest class; a source with one crop has ratio 1 and cannot be balanced):")
display(imbalance)

### 1.2 Parcel centroids

Each dot is one parcel's mean pixel location. The labels give parcels and pixels per source region. The star is the Atok ROI.

In [ ]:
region_counts = external.region_counts(data)
plots.parcel_centroids(data, region_counts, atok, roi, paths.countries_zip, FIGURES)
display(region_counts[["parcels", "pixels"]])

**Region × crop coverage.** Parcel counts per source region and crop, development + test, with Atok's provisional fields underneath. Empty cells are crop-region combinations with no data at all: a model trained on that region has never seen that crop.

In [ ]:
coverage, dev_test = external.region_crop_parcels(data, atok)
coverage.to_csv(OUTPUT / "region_crop_parcels.csv")
plots.region_crop_coverage(coverage, dev_test, FIGURES)

### 1.3 Half-month coverage vs the Atok crop calendar

The top panel shows the share of Atok ROI pixels with a clear composite in each half-month (2019-2025 mean). The rows below show:

- **Atok calendar:** each labelled crop season from the KML.
- **Sen4AgriNet and FranceCrops:** the share of parcels whose 8-half-month window covers each half-month (window starts from notebook 04's onset detection).
- **CropHarvest:** window start dates were not saved. For the JECAM rows (Madagascar, Senegal), the window is approximated as 8 half-months from the recorded planting date. The other CropHarvest regions have no dates, so they cannot be placed on this axis.

Each row is tagged by evidence type. **Assumed** rows come from the undated advisor calendar. **Observed** rows come from onset detection on the imagery. **Approximate** rows come from a recorded planting date plus 4 months. Months are calendar months: Madagascar is in the southern hemisphere, so its seasons are about 6 months out of phase with Atok's, and section 1.4 adds a season-aligned climate distance.

In [ ]:
windows = external.external_windows(paths, data)  # notebook 04 windows behind the Sen4AgriNet/FranceCrops table
catalogue = external.cropharvest_catalogue(paths.cropharvest_labels)  # target-crop labels only
ch_rows, ch_match = external.match_cropharvest(data, catalogue)
ch_planting = external.cropharvest_planting(ch_match)
print(f"CropHarvest rows matched to a catalogue label within {external.MATCH_RADIUS_M} m: "
      f"{(ch_match['distance_m'] <= external.MATCH_RADIUS_M).mean():.0%}")

In [ ]:
clear_by_half = atok_data.clear_share_by_half_month(roi)
coverage_rows = external.window_coverage_rows(windows, ch_rows, ch_planting)
plots.half_month_coverage(coverage_rows, clear_by_half, FIGURES)
month_clear = atok_data.monthly_clear_share(clear_by_half)
print("Atok clear share by month:", {m: round(v, 2) for m, v in month_clear.items()})

### 1.4 Temperature profiles: source regions vs Atok

**Source regions.** Monthly mean daily maximum and minimum temperature from TerraClimate (2019-2024, the same years as the PAGASA record; about 4 km), sampled at every parcel centroid and averaged per source region.

**Atok target.**
- **With `pagasa_baguio_path`:** PAGASA Baguio monthly TMAX/TMIN (`data/climate/pagasa_baguio_monthly.csv`, averaged by month), lapse-rate adjusted (6.5 °C/km) from the station elevation to Atok's 2,255 m (about 5 °C cooler).
- **Without it:** a flagged fallback. TerraClimate at the Atok ROI, adjusted from its 4 km cell's mean elevation to 2,255 m. ERA5-Land is not used for Atok: its ~9 km cells blur the elevation even more.

The distance is Euclidean over the 24 monthly values (12 TMAX + 12 TMIN), and regions are ranked from closest to farthest.

In [ ]:
climate_points, message = climate.sample_terraclimate(paths, data, roi.lon, roi.lat)
print(message)
climate_profiles = climate.atok_target(climate_points, paths)
print(f"Points with TerraClimate values: {climate_profiles.valid_points} of {climate_profiles.all_points} "
      "(coastal points can fall outside the land mask)")
print("Atok target:", climate_profiles.note)

In [ ]:
distances = climate.distances(climate_profiles)
distances.to_csv(OUTPUT / "climate_distances.csv")
display(distances)

In [ ]:
plots.temperature_profiles(climate_profiles, distances, FIGURES)

**Season-aligned distance and monthly differences.** Comparing calendar months penalises southern-hemisphere regions for being out of phase, not for being a different climate. The season-aligned distance shifts southern-hemisphere profiles by 6 months before comparing. The heatmaps show each region's monthly difference from the Atok target (region minus Atok, °C).

In [ ]:
region_lat = data.groupby("source_region")["lat"].mean()
distances = climate.add_aligned_distances(distances, climate_profiles, region_lat)
distances.to_csv(OUTPUT / "climate_distances.csv")
display(distances[["climate_distance", "rank", "aligned_distance", "aligned_rank", "hemisphere", "mean_tmax_diff", "mean_tmin_diff"]])

In [ ]:
plots.temperature_differences(climate_profiles, distances, FIGURES)

### 1.5 Domain shift and temporal signatures

The model can only transfer if Atok's spectral-temporal trajectories resemble those it was trained on. The first figure compares sources within each crop: median and interquartile range of NDVI, NDRE and NDMI over the 8 half-month bins. The second compares crops within each source, to show how separable the crops are at all. Atok uses its provisional labels.

In [ ]:
plots.domain_shift_by_crop(data, atok, FIGURES)

In [ ]:
plots.signatures_by_source(data, atok, FIGURES)

**A. Atok against each external region, all nine variables.** Each value is a pixel's mean over its 8 half-months, so these panels show level and spread; the timing is in panel B. The boxplots compare every region with Atok. The heatmap gives the distance between Atok's distribution and each region's for the same crop: the Wasserstein distance in units of the external standard deviation, averaged over the crops the two share. 0 means identical distributions; larger means Atok lies further from what the model learned from that region.

In [ ]:
both_means, shift = experiments.atok_shift(data, atok, distances.index)
plots.feature_distributions(both_means, list(distances.index) + [experiments.ATOK], FIGURES)

In [ ]:
shift.to_csv(OUTPUT / "atok_vs_region_wasserstein.csv")
plots.atok_vs_region_shift(shift, FIGURES)

**B. Crop growth curves and separability.** The first figure shows the mean ± 1 SD of each crop over the 8 half-months, for the verified external data (pooled), for Sen4AgriNet and CropHarvest separately, and for Atok's provisional labels. Comparing within one source shows crop differences without the source confound.

The second figure measures how separable each pair of crops is in each half-month. It uses the Jeffries-Matusita (JM) distance on all 9 variables of that half-month, assuming Gaussian classes. JM runs from 0 to 2: values above about 1.9 are conventionally well separable, and values below 1 mean the classes overlap heavily. Read it to see which growth stage, if any, separates the crops.

In [ ]:
separability_groups = experiments.separability_groups(data, atok)
plots.crop_growth_curves(separability_groups, FIGURES)

In [ ]:
separability = experiments.crop_separability(separability_groups)
separability.to_csv(OUTPUT / "crop_separability_jm.csv", index=False)
plots.crop_separability(separability, separability_groups, FIGURES)
display(separability.pivot_table(index=["group", "pair"], values="JM", aggfunc=["min", "max", "mean"]).round(2))

## 2. Preprocessing QC (Obj 2)

Preprocessing was done and audited in notebooks 01-05: reference extraction, Cloud Score+ masking, bilinear B5/B11 resampling, onset timing checks, complete-window selection and feature construction. See `docs/methodology.md` and `docs/workflow.md`. The tables below summarise the notebook 04 audits for the run that produced the Sen4AgriNet/FranceCrops table. CropHarvest was screened in notebook 03 and has no equivalent audit in that run.

In [ ]:
audits = external.preprocessing_audits(paths)
print("Automatic timing checks (parcel-season candidates, Sen4AgriNet + FranceCrops):")
display(audits["timing"]["automatic_timing_status"].value_counts().rename("candidates").to_frame())

In [ ]:
print("Cycle selection status after the timing checks:")
display(audits["selection"].groupby(["source_dataset", "crop", "status"]).size().unstack(fill_value=0))

In [ ]:
print("Usable pixels per crop (complete windows) and the per-crop cap:")
display(audits["usable"])

## 3. Dataset specification (Obj 3a)

### 3.1 Specification

**Decision on "others".** Built-up and non-crop land become one `others` class, digitised from NAMRIA ELUM CY-2023 non-crop polygons and checked against drone imagery where available. Other crops (lettuce, radish, seedbeds) need their own `other_crop` class once labelled. Neither is trained yet: the external tables have no such rows. Until they are, the maps use the ELUM Crops mask plus an **unknown** label (section 7) for low-probability and out-of-distribution pixels. That reduces wrong labels, but it does not replace these classes.

In [ ]:
spec_table = report.dataset_specification()  # text in report.DATASET_SPECIFICATION
spec_table.to_csv(OUTPUT / "dataset_specification.csv", index=False)
display(spec_table.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

### 3.2 Exclusion funnel and record-level exclusions

Each panel follows one sampling unit (labels, parcel-seasons, parcels, pixels, pixel-windows) and shows the share retained relative to the first step of that unit. The CropHarvest catalogue records get a record-level status, saved with their catalogue index in `cropharvest_record_exclusions.csv`. For the records that reached the training table, the reason other records were dropped at the screening and sampling step was not logged at export. That step is reported as unlogged rather than guessed.

In [ ]:
funnel = report.exclusion_funnel(catalogue, ch_match, ch_rows, audits, data, tables, atok, atok_windows)
funnel.to_csv(OUTPUT / "exclusion_funnel.csv", index=False)
display(funnel)

In [ ]:
plots.exclusion_funnel(funnel, FIGURES)

In [ ]:
records = report.cropharvest_record_status(catalogue, ch_match)
records.to_csv(OUTPUT / "cropharvest_record_exclusions.csv", index=False)
print("CropHarvest catalogue records by status (before) and crop:")
display(pd.crosstab(records["status"], records["target_crop"], margins=True))

In [ ]:
print("Crop distribution before and after (labels in the catalogue vs parcels in the training table):")
display(report.crop_distribution_before_after(records, ch_rows))

## 4. External models and diagnostics (Obj 3b)

### 4.1 Same-predictor comparison on the locked test set

Both classifiers run on both predictor sets, so model choice and feature representation can be separated. The **RF on interval predictors is the prespecified prototype**. The **LR on shape predictors was selected by v1's grouped CV**. No configuration below was chosen using the test set. Every table has two baselines: always predicting the most common class, and the expected score of guessing with the class shares. **Balanced accuracy** is the mean producer's accuracy (recall) over crops. Producer's accuracy is the share of a crop that is found; user's accuracy is the share of mapped pixels of a crop that are right.

In [ ]:
FOLDS = models.grouped_folds(development)  # the same folds as v1's CV
pooled, test_results = models.fit_pooled_models(development, test)  # models.MODEL_SPECS
test_results.to_csv(OUTPUT / "external_test_results.csv", index=False)
display(test_results)

### 4.2 Grouped CV from v1 (development parcels)

v1's grouped 5-fold CV (same folds) for every design × model, mean ± sd macro-F1. Read along a row to compare models on the same predictors, and down a column to compare predictor sets for the same model.

In [ ]:
v1_cv = pd.read_csv(paths.v1 / "cv_summary.csv")
v1_cv = v1_cv[(v1_cv["grouping"] == "split_unit") & (v1_cv["weighted"].astype(str) == "True")]
display(v1_cv.assign(score=v1_cv["macro_f1_mean"].map("{:.3f}".format) + " ± " + v1_cv["macro_f1_std"].map("{:.3f}".format))
        .pivot(index="design", columns="model", values="score"))

### 4.3 RF tuning and overfitting (grouped CV)

The prototype's settings were prespecified, not tuned. The grid below covers minimum leaf size, features per split and maximum depth on the interval predictors. Every configuration uses the same 5 grouped folds and 2 seeds, with 500 trees each to keep the run short. Training and validation macro-F1 are reported together, so the gap (overfitting) is visible. The best and the prototype configurations are then rerun with **source-balanced** weights. The best configuration by validation macro-F1 is refitted with 1,000 trees and scored once on the locked test set, as a reported alternative to the prototype.

In [ ]:
tuning = experiments.tuning_grid(development, FOLDS, models.PROTOTYPE_PARAMS)  # experiments.TUNING_GRID
tuning.to_csv(OUTPUT / "rf_tuning_grid.csv", index=False)
display(tuning)
TUNED_PARAMS = experiments.best_params(tuning)
print("Best by grouped CV:", TUNED_PARAMS)

In [ ]:
weighting = experiments.weighting_comparison(development, FOLDS, {"prototype": models.PROTOTYPE_PARAMS, "tuned": TUNED_PARAMS})
weighting.to_csv(OUTPUT / "rf_weighting_comparison.csv", index=False)
display(weighting)

In [ ]:
TUNED = "RF, interval (tuned by grouped CV)"
pooled[TUNED] = (models.fit_rf(development, INTERVAL, **TUNED_PARAMS), INTERVAL)
test_results = models.append_test_row(test_results, models.test_row(TUNED, pooled[TUNED][0], test, INTERVAL))
test_results.to_csv(OUTPUT / "external_test_results.csv", index=False)
display(test_results.tail(1))

### 4.4 Learning curves by crop

These use the prototype's settings, including 1,000 trees, and 3 seeds per training fraction. Training units are subsampled within each crop in every grouped fold. The panels show validation macro-F1 against training macro-F1, and validation recall per crop. If validation keeps rising as parcels are added, more data should help. If it flattens while the training-validation gap stays large, more data from the same sources will not close the gap: the cause is more likely source differences or label inconsistency.

In [ ]:
curve = experiments.learning_curve(development, FOLDS)
curve.to_csv(OUTPUT / "learning_curve.csv", index=False)
curve_means = curve.groupby("fraction").mean()
display(curve_means[["train_parcels", "train_macro_F1", "validation_macro_F1"] + [f"recall_{c}" for c in CLASSES]])

In [ ]:
plots.learning_curves(curve, FIGURES)

### 4.5 Grouped permutation importance and feature-family ablation

Predictors in neighbouring half-months and related indices are strongly correlated, so permuting single columns understates shared information, and summing those drops is misleading. Here whole groups are permuted together on each grouped validation fold (5 repeats):

- **one variable across all 8 bins**, and
- **one bin across all 9 variables**.

Negative values mean shuffling the group did not hurt, or slightly helped, on that fold: the group carries no usable signal there. The ablation refits the RF without each variable family and reports the change in validation macro-F1.

In [ ]:
importance, ablation = experiments.importance_and_ablation(development, FOLDS)
importance.to_csv(OUTPUT / "grouped_permutation_importance.csv", index=False)
ablation.to_csv(OUTPUT / "family_ablation.csv", index=False)
plots.importance_ablation(importance, ablation, FIGURES)

### 4.6 Calibration, abstention threshold and out-of-distribution check

The top-class probability of a Random Forest is not automatically a calibrated probability of being right. Reliability diagrams and expected calibration error (ECE) are computed on out-of-fold predictions for the development parcels, and on the locked test set; both have verified external labels. The selective-prediction curve shows accuracy against the share of predictions kept. The **abstention threshold** is the lowest top-class probability whose out-of-fold kept predictions reach the target selective accuracy. It is set on development data and then checked on the test set.

The **out-of-distribution** check is the mean distance to the 5 nearest development samples in standardised interval-feature space. A sample is flagged when it is farther than the chosen quantile of the external test samples' distances. Calibration on Atok cannot be assessed until verified Atok labels exist.

In [ ]:
oof = experiments.out_of_fold(development, FOLDS)  # prototype RF and v1's LR
reliability_curves, calibration = experiments.calibration_check(development, test, pooled, oof)
calibration.to_csv(OUTPUT / "calibration.csv", index=False)
abstention = experiments.raw_abstention(development, test, pooled, oof[PROTOTYPE])

In [ ]:
distance_to_training = models.DistanceToTraining(development)
test_distance = distance_to_training(test)
OOD_THRESHOLD = float(np.quantile(test_distance, OOD_QUANTILE))
atok_distance = distance_to_training(atok)
atok["ood"] = atok_distance > OOD_THRESHOLD

In [ ]:
plots.calibration_selective_ood(reliability_curves, test_distance, atok_distance, OOD_THRESHOLD, FIGURES)

In [ ]:
display(calibration)
print("Abstention threshold:", abstention)
print(f"Out-of-distribution: {atok['ood'].mean():.0%} of Atok pixel-windows vs {1 - OOD_QUANTILE:.0%} of the external test set by construction")

### 4.6b Fitting a calibration, and whether the threshold transfers to new regions

Section 4.6 only **measured** calibration. Here the prototype's probabilities are **calibrated**, by one of two methods fitted on the grouped out-of-fold predictions for the development parcels:

- **temperature scaling:** one parameter; it never changes the predicted crop;
- **matrix scaling:** a multinomial logistic map from log-probabilities.

**No calibration** (the raw probabilities) is the third candidate. Whichever of the three has the lowest cross-fitted ECE on development data is used; the test set plays no part in that choice. If neither fitted method beats the raw probabilities, the raw ones are kept, and that is a result: fitting was tried and did not help. To evaluate it honestly:

- **cross-fitted:** each fold's out-of-fold predictions are recalibrated by a calibrator fitted on the other folds only;
- **held-out:** the final calibrator, fitted on all out-of-fold predictions, is scored once on the locked test set.

The abstention threshold is then chosen on the cross-fitted calibrated probabilities. The target is the same: kept predictions reach the target selective accuracy.

**Does the threshold transfer?** Each region in turn is held out. The model, its out-of-fold predictions, the calibrator and the threshold all come from the other regions only. The threshold is then applied to the unseen region. If the kept predictions in unseen regions fall well short of the target, a threshold set on external data cannot be trusted in Atok either. Atok's calibration remains unknown until verified Atok labels exist.

In [ ]:
calibration_fit = experiments.fit_calibration(development, test, pooled, FOLDS, oof[PROTOTYPE])
CAL_METHOD = calibration_fit["method"]
calibration_fit["table"].to_csv(OUTPUT / "calibration_fitted.csv", index=False)
print("Calibration method chosen by cross-fitted ECE on development data:", CAL_METHOD,
      f"(temperature {calibration_fit['calibrator'].temperature:.2f})" if CAL_METHOD == "temperature" else "",
      "- no fitted calibration improved on the raw probabilities, so they are kept" if CAL_METHOD == "none" else "")
display(calibration_fit["table"])
print(f"Threshold on calibrated probabilities: {calibration_fit['threshold']} (target {config.TARGET_SELECTIVE_ACCURACY:.0%} "
      f"reached out of fold: {calibration_fit['reached']}); locked test: keeps {calibration_fit['test_coverage']:.0%} "
      f"at {calibration_fit['test_accuracy_kept']:.0%} accuracy")

In [ ]:
threshold_transfer, transfer_curves = experiments.threshold_transfer(data, distances.index, CAL_METHOD)
threshold_transfer.to_csv(OUTPUT / "threshold_transfer_by_region.csv", index=False)
display(threshold_transfer)
print(f"Threshold transfers (kept accuracy >= {config.TARGET_SELECTIVE_ACCURACY:.0%}) in "
      f"{int(threshold_transfer['meets target'].sum())} of {len(threshold_transfer)} unseen regions")

In [ ]:
atok_calibrated = models.apply_calibrator(calibration_fit["calibrator"], models.aligned_proba(pooled[PROTOTYPE][0], atok, INTERVAL))
plots.threshold_transfer(calibration_fit["test_calibrated"], test, transfer_curves, threshold_transfer, atok_calibrated, atok, FIGURES)

### 4.7 Transfer to unseen regions

The locked test set mixes regions that are also in training, so it does not measure transfer to a new region. Two experiments do.

- **Leave one region out:** train the prototype RF on every other region (all their parcels) and test on the held-out region. Crops that the remaining regions never contain cannot be predicted and are listed.
- **Single-region transfer:** train on one region (regions with at least 2 crops) and test on every other region, plus provisional Atok. Scores over 1, 2 or 3 shared crops are different tasks, so they are shown on separate panels. The comparable measure is the one-vs-rest AUC per crop that follows.

In [ ]:
REGIONS = list(distances.index)
loro = experiments.leave_one_region_out(data, REGIONS)
loro.to_csv(OUTPUT / "leave_one_region_out.csv", index=False)
display(loro)

In [ ]:
transfer = experiments.transfer_matrix(data, atok, REGIONS)
transfer["matrix"].to_csv(OUTPUT / "transfer_matrix_balanced_accuracy.csv")
transfer["shared_count"].to_csv(OUTPUT / "transfer_matrix_shared_crops.csv")
plots.transfer_matrix(transfer, FIGURES)

**One-vs-rest transfer per crop (comparable across pairs).** For each crop, a train → test pair qualifies only when **both** regions contain that crop and at least one other crop. The other crops in the test region are then valid negatives. The score is the ROC AUC of the model's probability for that crop: chance is 0.5 whatever the number of crops, and it needs no threshold. These are the transfer scores used for the similarity analysis (C) below.

In [ ]:
similarity = climate.RegionSimilarity(climate_profiles, data, roi.lon, roi.lat)  # season-aligned climate and km
ovr = experiments.one_vs_rest_transfer(data, atok, transfer, similarity)
ovr.to_csv(OUTPUT / "transfer_one_vs_rest_auc.csv", index=False)
plots.one_vs_rest_auc(ovr, transfer, FIGURES)
display(experiments.auc_summary(ovr))

**C. Does similarity predict transfer?** Each point is one train → test → crop combination from the one-vs-rest table, so every point is scored on the same scale (AUC, chance 0.5). The x-axis is either the season-aligned climate distance (the TMAX/TMIN measure of section 1.4) or the great-circle distance between the region centroids. If similarity drives transfer, AUC should fall as distance grows. The points share regions and crops, so they are not independent: ρ is descriptive.

In [ ]:
pair_stats = experiments.similarity_correlations(ovr)
pair_stats.to_csv(OUTPUT / "transfer_similarity_correlations.csv", index=False)
plots.transfer_vs_similarity(ovr, pair_stats, FIGURES)
display(pair_stats)

## 5. Climate-matching experiment

### 5.1 Primary test: external target regions with verified labels

The climate question is tested where the labels can be trusted. Each external region with at least two crops becomes a **target** in turn. Training sets are combinations of the other regions. A combination is eligible only if it holds at least `MIN_PARCELS_PER_CROP` parcels of every crop the target contains, so every set is scored on the same task for a given target.

For each eligible set:

- an RF (prototype settings, 500 trees) is trained on all parcels of those regions and scored on all parcels of the target: balanced accuracy over the target's crops;
- the set's climate distance to the target is the parcel-weighted mean of its regions' season-aligned distances; the geographic distance is computed the same way.

Within each target, Spearman's ρ between climate distance and transfer, and a slope adjusted for training size, show whether closer climates transfer better. The robustness check reruns ρ after dropping every set that contains any one source region. **If the effect is real, ρ should be negative in most targets and survive removing any single source.** Atok is not used here.

In [ ]:
TARGETS, external_sets = experiments.climate_external_targets(data, REGIONS, similarity)
external_sets.to_csv(OUTPUT / "climate_external_targets.csv", index=False)
target_stats = experiments.climate_target_stats(data, external_sets, REGIONS)
target_stats.to_csv(OUTPUT / "climate_external_summary.csv", index=False)
display(target_stats)

In [ ]:
tested = target_stats.dropna(subset=["rho_climate"])
negative = tested["rho_climate"] < 0
print(f"Targets with enough eligible sets: {len(tested)}; climate rho negative in {int(negative.sum())}, "
      f"significant (p < 0.05) and negative in {int((negative & (tested['p_climate'] < 0.05)).sum())}; "
      f"negative after dropping any single source in {int(tested['negative_after_every_drop'].sum())}")

In [ ]:
plots.climate_external_targets(data, TARGETS, external_sets, target_stats, FIGURES)

### 5.2 Exploratory application: Atok

The same idea applied to Atok, **for exploration only**. These scores measure agreement with undated advisor rotations on 5 field-crops, so they cannot establish whether climate similarity helps (5.1 does that). They are kept because they choose the climate-ranked training set used for the second Atok map model, and to show how the provisional labels respond.

Sets are eligible if they hold at least `MIN_PARCELS_PER_CROP` development parcels of every crop. The climate-ranked set is the smallest prefix of the climate ranking that is eligible. It is compared with all other eligible sets, and with size-matched random parcels.

In [ ]:
dev_parcels = experiments.development_parcels(development, REGIONS)
eligible_sets = experiments.eligible_region_sets(dev_parcels)
print(f"Eligible region sets (>= {config.MIN_PARCELS_PER_CROP} development parcels of every crop): "
      f"{len(eligible_sets)} of {2 ** len(REGIONS) - 1}")
display(dev_parcels.assign(eligible_alone=(dev_parcels >= config.MIN_PARCELS_PER_CROP).all(axis=1)))

In [ ]:
CLIMATE_SET = experiments.ranked_prefix(dev_parcels, list(distances.sort_values("climate_distance").index))
ALIGNED_SET = experiments.ranked_prefix(dev_parcels, list(distances.sort_values("aligned_distance").index))
print("Climate-ranked set (calendar months):", CLIMATE_SET)
print("Climate-ranked set (season-aligned):", ALIGNED_SET)

In [ ]:
region_sets = experiments.atok_region_sets(development, atok, eligible_sets, dev_parcels, distances, CLIMATE_SET, ALIGNED_SET)
region_sets.to_csv(OUTPUT / "climate_region_sets.csv", index=False)
correlations = experiments.region_set_correlations(region_sets)
correlations.to_csv(OUTPUT / "climate_correlations.csv", index=False)
display(correlations)

In [ ]:
climate_rows = development[development["source_region"].isin(CLIMATE_SET)]
control_scores = experiments.size_matched_controls(development, atok, climate_rows)
climate_score = float(region_sets.loc[region_sets["is_climate_set"], "atok_balanced_acc"].iloc[0])
print(f"Climate-ranked set Atok balanced accuracy {climate_score:.3f}; size-matched random controls "
      f"{np.mean(control_scores):.3f} ± {np.std(control_scores, ddof=1):.3f} "
      f"(climate set above {np.mean(np.array(control_scores) < climate_score):.0%} of draws)")
display(region_sets.head(10))

In [ ]:
plots.climate_experiment(region_sets, correlations, control_scores, climate_score, FIGURES)

In [ ]:
CLIMATE_LABEL = " + ".join(r.split(": ")[1] for r in CLIMATE_SET)
CLIMATE_MODEL = f"RF, interval, climate set ({CLIMATE_LABEL})"
climate_model, climate_calibrator, CLIMATE_THRESHOLD = models.calibrated_model(climate_rows, CAL_METHOD)
print(f"Climate-set model: calibrated threshold {CLIMATE_THRESHOLD}")
test_results = models.append_test_row(test_results, models.test_row(CLIMATE_MODEL, climate_model, test, INTERVAL))
test_results.to_csv(OUTPUT / "external_test_results.csv", index=False)
display(test_results.tail(1))

**Robustness: is it climate, or one source?** Region sets overlap, so the p-values above treat 87 non-independent sets as independent. In addition, a parcel-weighted distance is dominated by whichever region contributes the most parcels. The check below compares sets with and without each region, and recomputes the correlation after dropping every set that contains that region. If the climate effect is real, the correlation should survive removing any single region.

In [ ]:
robustness = experiments.drop_one_region(region_sets, REGIONS)
robustness.to_csv(OUTPUT / "climate_robustness.csv", index=False)
display(robustness)

## 6. Atok evaluation (provisional labels)

**These labels are not ground truth.** They are advisor-described, undated rotations, repeated every year. There are 2 carrot, 2 cabbage and 1 potato field. The 8,903 pixel-windows are not independent: they are repeated pixels and years of 5 field-crop combinations. Read every number below as agreement with an assumed calendar on 5 field-crops.

### 6.1 Transferred models against baselines

Carrot is the majority of the Atok samples, so **always answering carrot** is the baseline to beat on overall accuracy. On balanced accuracy, the baseline is 1/3.

In [ ]:
ATOK_MODELS = {**pooled, CLIMATE_MODEL: (climate_model, INTERVAL)}
display(atok_units)
atok_results, atok_predictions = experiments.atok_evaluation(atok, ATOK_MODELS)
atok_results.to_csv(OUTPUT / "atok_transfer_results.csv", index=False)
display(atok_results)

In [ ]:
plots.atok_confusion_matrices(atok, atok_predictions, FIGURES)

### 6.2 Clear-data sensitivity

Atok windows may have up to 2 interpolated half-months, while the external windows are complete. The same models are rescored on Atok samples with at least 5, 6 (the default), 7 and 8 clear composites; 8 means no interpolation, matching the external data. The window length (8 half-months) cannot be varied here because the external tables were exported with 8 bins.

In [ ]:
sensitivity = experiments.clear_sensitivity(atok_all, ATOK_MODELS, [PROTOTYPE, LR_SHAPE, CLIMATE_MODEL])
sensitivity.to_csv(OUTPUT / "atok_clear_sensitivity.csv", index=False)
display(experiments.sensitivity_table(sensitivity))

### 6.3 Domain shift on Atok

The first panel puts Atok samples and the external development samples in the same two principal components, fitted on the external data. The second shows the share of Atok samples flagged out-of-distribution (4.6), by field and crop. A model is extrapolating whenever most of its inputs are out of distribution.

In [ ]:
ood_table = plots.atok_domain_shift(development, atok, distance_to_training, FIGURES)
display(ood_table)

### 6.4 Failure analysis

This finds which field, season and year combinations fail, and whether failure follows cloud cover (clear composites) or distance from the training data. Each row is one field, crop and calendar window.

In [ ]:
FAILURE_MODELS = {"pooled RF": PROTOTYPE, "climate-set RF": CLIMATE_MODEL}
failure, by_window = experiments.failure_table(atok, atok_predictions, FAILURE_MODELS)
by_window.to_csv(OUTPUT / "atok_failure_by_window.csv")
display(by_window)

In [ ]:
plots.atok_failure_analysis(failure, FAILURE_MODELS, FIGURES)

### 6.5 Weak-label fitting diagnostic (not a measure of generalisation)

**Field-independent validation is unavailable:** it needs at least 2 fields of every crop, and potato has 1. This section therefore splits field-season-years 70/30 by crop, so **the same physical fields, and the same repeated calendar assumption, appear in training and test**. It only shows whether an RF can fit the weak labels at all. It must not be quoted as accuracy, nor compared with the transferred models as if it were. The transferred models are scored on the same 30% for reference.

In [ ]:
benchmark = experiments.local_benchmark(atok, ATOK_MODELS)
FIELD_SPLIT = benchmark["field_split"]
print("Labelled fields per crop:", benchmark["fields_per_crop"].to_dict(), "| field-independent split possible:", FIELD_SPLIT)
if not FIELD_SPLIT:
    print("Field-independent validation: UNAVAILABLE. The split below is a weak-label fitting diagnostic only.")

In [ ]:
benchmark["table"].to_csv(OUTPUT / "local_benchmark.csv", index=False)
print(f"Test field-season-years: {len(benchmark['test_groups'])}; test pixel-windows per crop: {benchmark['test_crops']}")
display(benchmark["table"])

## 7. Atok maps (exploratory)

These maps show what the models would produce, **not verified crop locations**. Two models are mapped: the prespecified pooled RF and the climate-set RF (section 5). Each is shown for the best- and the worst-coverage year, in the three calendar windows. Prediction is limited to the ELUM CY-2023 Crops mask. A pixel is labelled **unknown** when its top-class probability is below the abstention threshold from 4.6, or when it is out of distribution. Otherwise it gets one of the three crops, which may be wrong for any other crop or fallow land. GeoTIFFs: band 1 is the class code (1 carrot, 2 cabbage, 3 potato, 4 unknown), band 2 the top-class probability (uncalibrated for Atok), band 3 the distance to the training data.

### Cropland mask

The mask is the NAMRIA ELUM CY-2023 **Crops** polygons within the ROI. It is compared with notebook 09's unsupervised active-cropland map when that map is available.

In [ ]:
cropland = atok_data.cropland_mask(roi, paths.landuse)
print(f"ELUM 2023 Crops: {cropland.sum():,} of {roi.roi_mask.sum():,} ROI pixels ({cropland.sum() / roi.roi_mask.sum():.0%}); "
      f"{cropland.sum() * 0.01:,.0f} ha")
activity = atok_data.compare_land_activity(roi, cropland, paths.land_activity)
if activity:
    print(f"Notebook 09 calls {activity['elum_crops_active']:.0%} of the ELUM Crops pixels active cropland; "
          f"{activity['active_inside_elum_crops']:.0%} of its active cropland lies inside ELUM Crops")

In [ ]:
mappable = maps.mappable_share(roi, cropland)
BEST_YEAR, WORST_YEAR = int(mappable["mean"].idxmax()), int(mappable["mean"].idxmin())
print(f"Share of the cropland mask with >= {MIN_CLEAR} of 8 clear composites:")
display(mappable.style.format("{:.0%}"))
print(f"Best-coverage year {BEST_YEAR}, worst {WORST_YEAR}")

In [ ]:
MAP_MODELS = {  # name -> (file slug, calibrated model, calibrated threshold)
    "Pooled RF (prespecified prototype)": (
        "pooled_rf", models.Calibrated(pooled[PROTOTYPE][0], calibration_fit["calibrator"]), calibration_fit["threshold"]),
    f"Climate-set RF ({CLIMATE_LABEL})": (
        "climate_set_rf", models.Calibrated(climate_model, climate_calibrator), CLIMATE_THRESHOLD),
}
predictor = maps.MapPredictor(roi, cropland, distance_to_training, OOD_THRESHOLD)
map_raw, map_summary = maps.predict_maps(predictor, MAP_MODELS, (BEST_YEAR, WORST_YEAR), OUTPUT)  # also writes GeoTIFFs
background = plots.map_background(roi, cropland, maps.never_clear(roi))

In [ ]:
plots.crop_maps(map_raw, MAP_MODELS, BEST_YEAR, BEST_YEAR, roi, background, FIGURES)

In [ ]:
plots.crop_maps(map_raw, MAP_MODELS, WORST_YEAR, BEST_YEAR, roi, background, FIGURES)

In [ ]:
map_summary.to_csv(OUTPUT / "atok_map_summary.csv", index=False)
display(map_summary)

### 7.0 Why pixels are rejected, and what coverage a threshold would give

Two separate mechanisms make a pixel **unknown**:

- **uncertainty:** the calibrated top-class probability is below the model's threshold (4.6b);
- **out of distribution:** the pixel's features are farther from the training data than 95% of the external test samples.

The first table and maps split every rejected pixel into *uncertain only*, *out of distribution only* or *both*, for the best-coverage year. The operating-point curves show how much of the mappable cropland would be classified at other thresholds, with and without the out-of-distribution check. The proposed **minimum coverage requirement** is 50% of mappable cropland per window. The threshold behind these maps was set on external data, and 4.6b tests whether it holds in unseen regions; Atok cannot be checked without verified labels.

In [ ]:
rejection_causes, cause_masks = maps.rejection_causes(map_raw, MAP_MODELS, BEST_YEAR)
rejection_causes.to_csv(OUTPUT / "atok_rejection_causes.csv", index=False)
display(rejection_causes)

In [ ]:
plots.rejection_cause_maps(cause_masks, MAP_MODELS, BEST_YEAR, roi, background, FIGURES)

In [ ]:
operating = maps.operating_points(map_raw, MAP_MODELS, BEST_YEAR)
operating.to_csv(OUTPUT / "atok_operating_points.csv", index=False)
plots.operating_points(operating, MAP_MODELS, BEST_YEAR, FIGURES)

### 7.1 Where the two models disagree

For the best-coverage year: green where both models give the same crop, red where they give different crops, yellow where either says unknown. The transition tables give pixel counts, not shares.

In [ ]:
agreement, transitions, agreement_masks = maps.model_agreement(map_raw, MAP_MODELS, BEST_YEAR)
plots.model_disagreement(agreement_masks, BEST_YEAR, roi, background, FIGURES)

In [ ]:
agreement.to_csv(OUTPUT / "atok_map_agreement.csv", index=False)
display(agreement)
for window, table in transitions:
    print(window)
    display(table)

### 7.2 Where the models are wrong on the labelled fields

**No verified crop polygons exist yet.** These maps zoom in on the provisional KML fields (B1, Area 2, Area 3) and score every interior pixel against the calendar crop for that window. Each pixel falls in one category:

- **correct** (blue): the model gives the calendar crop;
- **wrong crop:** the model gives a different crop, with the colour of the crop it predicted;
- **unknown:** abstained (low probability or out of distribution);
- **too cloudy:** fewer than 6 of 8 clear composites.

The first figure shows the best-coverage year, one column per calendar window. The second shows each pixel's share of correct windows across all years 2019-2025 (only windows where it was clear enough), which tells apart consistent failures from one-off ones. Once verified polygons are digitised, they replace the calendar labels here.

In [ ]:
called, unknown = maps.labelled_field_calls(atok, MAP_MODELS)  # same calibrated models and thresholds as the maps
field_errors = maps.field_errors(atok, called, unknown, BEST_YEAR)
plots.field_errors(atok, called, unknown, BEST_YEAR, roi, maps.labelled_fields(), FIGURES)

In [ ]:
field_errors.to_csv(OUTPUT / f"atok_field_errors_{BEST_YEAR}.csv", index=False)
display(field_errors)

In [ ]:
per_pixel_shares, consistency = maps.field_consistency(atok, called, unknown)
plots.field_consistency(per_pixel_shares, roi, maps.labelled_fields(), FIGURES)
consistency.to_csv(OUTPUT / "atok_field_consistency.csv", index=False)
display(consistency)

## 8. Limitations and requirements (Obj 4)

Each limitation has an evidence number computed in this notebook (or read from v1), the requirement that would address it, and a priority.

In [ ]:
v1_transfer = json.loads((paths.v1 / "transfer_metrics.json").read_text())
results = {  # evidence for the limitations, criteria and summary tables
    "atok": atok, "atok_units": atok_units, "grid": roi, "cropland": cropland, "atok_results": atok_results,
    "month_clear": month_clear, "mappable": mappable, "best_year": BEST_YEAR, "worst_year": WORST_YEAR,
    "test_results": test_results, "test_parcels": test["partition_group"].nunique(),
    "grid_table": tuning, "prototype_gap": tuning.loc[tuning["prototype"], "gap"].iloc[0], "tuned_params": TUNED_PARAMS,
    "calibration": calibration_fit, "abstention": abstention, "threshold_transfer": threshold_transfer,
    "rejection_causes": rejection_causes, "target_stats": target_stats, "agreement": agreement,
    "france_accuracy": v1_transfer["france_to_cropharvest"]["by_region"]["France"]["accuracy"],
    "pagasa_elevation_m": paths.pagasa_elevation_m, "field_split": FIELD_SPLIT,
    "target_note": climate_profiles.note, "climate_set": CLIMATE_SET, "aligned_set": ALIGNED_SET,
    "eligible_sets": len(eligible_sets), "correlations": correlations, "climate_score": climate_score,
    "control_scores": control_scores, "map_models": MAP_MODELS, "ood_threshold": OOD_THRESHOLD,
}

In [ ]:
limitations = report.limitations(results)
limitations.to_csv(OUTPUT / "limitations_requirements.csv", index=False)
display(limitations.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

## 9. Pilot acceptance criteria (proposed)

These criteria are proposed **for the verified pilot**: fix them before that evaluation, not after. They are shown against the current provisional results only to make the gap explicit. A model would count as a working Atok classifier only if it meets every row on verified, field-independent data.

The pilot also needs things this notebook cannot provide:

- **Representative sampling:** plots across elevation, slope, aspect, field size and mixed cropping.
- **An end-to-end run:** imagery ingestion → inference → map → interpretation by agricultural staff.
- **A failure analysis on verified fields:** clouds, phenology, shadows, crop similarity and label quality.

In [ ]:
criteria = report.acceptance_criteria(results)
criteria.to_csv(OUTPUT / "acceptance_criteria.csv", index=False)
display(criteria)
print(f"Criteria met: {int(criteria['met'].sum())} of {len(criteria)}")

## Summary

From the run on 2026-11-09 (PAGASA Baguio target, lapse-adjusted −4.9 °C to 2,255 m). **This is an exploratory feasibility study. None of the 12 proposed pilot acceptance criteria (section 9) is met, and 2 (map coverage and Atok calibration) cannot even be assessed without verified Atok labels: there are 0 verified fields, and only 2 carrot, 2 cabbage and 1 potato provisional fields.** Field-independent validation is unavailable.

**External data (verified labels)**
- **Same predictors, different classifiers (4.1, 4.2):** interval predictors beat shape predictors for both models. On the locked test set, LR on interval predictors scores macro-F1 0.657 (95% CI 0.475-0.776) against 0.581 (0.403-0.704) for the prespecified RF; v1's CV gave the same order. The prototype stays prespecified. The intervals overlap widely.
- **Overfitting (4.3, 4.4):** the prototype's grouped-CV gap is 0.46 (training 0.98, validation 0.52). A minimum leaf of 25 cuts the gap to 0.21 at the same validation score (0.53), so the gap is memorisation, not useful fit. Source-balanced weights add ≤ 0.01. Validation flattens at about 0.52, and carrot recall stays near 0.25 at every training size.
- **Calibration (4.6, 4.6b):** the raw RF probabilities are already fairly well calibrated (ECE 0.039 out of fold, 0.081 on test). Temperature scaling (0.063) and matrix scaling (0.082) were both fitted and cross-validated, and both made calibration worse, so the raw probabilities are kept. The abstention threshold (0.66) keeps 33% of test predictions at 84% accuracy.
- **Threshold transfer (4.6b):** set on the other regions and applied to an unseen one, the threshold reaches 70% kept accuracy in **1 of 7** regions (Réunion). Elsewhere, kept accuracy is 0-28%. **A threshold set on external data does not carry over to a new region, so the one used for Atok has no validated meaning there.**
- **New regions (4.7):** leave-one-region-out balanced accuracy is 0.12-0.35 for most regions (0.75 for Réunion). On the comparable one-vs-rest measure (AUC, chance 0.5), single-region transfer averages 0.51 for carrot, 0.60 for cabbage and 0.60 for potato on external targets. On Atok it is 0.36-0.41, **below chance** for all three crops.

**Climate matching (5)**
- **Primary test, external targets with verified labels (5.1):** across 5 target regions, the climate-distance correlation is negative in 4 and significant in 2 (Réunion ρ −0.49, Martinique −0.54). It survives dropping any single source region in **only one target** (Réunion): in the others it flips sign when one source is removed. Over all single-region pairs, neither climate distance (ρ −0.14) nor geographic distance (+0.09) predicts one-vs-rest transfer. **The data do not show a general climate effect.** The Réunion result is the one robust exception, and it rests on one target.
- **Atok (5.2, exploratory only):** the apparent climate correlation across Atok region sets (ρ −0.72 to −0.80) is a Sen4AgriNet Catalonia effect. It vanishes without Catalonia (ρ +0.17 to +0.19). The climate-ranked set scores 0.361 balanced accuracy, barely above chance (0.333).

**Atok (provisional labels, 5 field-crops)**
- **Baselines (6.1):** always answering carrot gets OA 0.575; every transferred model gets 0.20-0.27. The best balanced accuracy (0.361, climate-set RF) comes from calling most samples cabbage.
- **Clear-data rule (6.2):** requiring 8 of 8 clear composites (no interpolation) does not help: balanced accuracy falls for both RFs. **Domain shift (6.3):** 22% of Atok samples are out of distribution, against 5% of external test samples.
- **Weak-label fitting diagnostic (6.5):** an RF trained on 70% of Atok's field-season-years fits the other 30% at OA 0.70, balanced accuracy 0.51, with cabbage recall 0.08. This shares fields between training and test, so it is **not** an accuracy estimate.

**Diagnostic visuals**
- **Spectral similarity is not climate similarity (1.5):** Atok's spectral distributions are closest to Catalonia and FranceCrops (about 0.43 SD), not to the climate-matched regions (Réunion 0.97, Madagascar 0.81).
- **Separability:** within CropHarvest the crops are fairly separable (Jeffries-Matusita 1.4-1.8). Carrot and potato overlap within Sen4AgriNet (0.67-1.10), and pooling sources blurs every pair.

**Maps and rejection (7)**
- **Pooled RF (2024):** with its threshold (0.66), it classifies only 1-22% of mappable cropland. Most rejections are uncertainty (50-69% of pixels), with out-of-distribution adding up to 49% in Jul-Oct.
- **Climate-set RF:** its in-source threshold falls to the floor (0.34) because it is accurate within its own sources. So it rejects only out-of-distribution pixels and classifies 51-90%, without any evidence that those labels are right in Atok.
- **Coverage:** the operating-point curves show what lower thresholds would give. No threshold can be validated without verified Atok labels.
- **Agreement:** the two models give the same crop on 25-86% of the pixels both classify.
- **Labelled fields (2024):** 84% (pooled) and 35% (climate set) of label-field pixels are never correct in any window from 2019 to 2025.

**What would change the conclusions:** dated, field-checked labels on at least 5 fields per crop, with a field-independent holdout, a threshold and calibration chosen on that verified data, and a minimum coverage of 50%. Also needed: each crop from more than one source and more regions, so climate can be separated from source; and Sentinel-1 for the May-August gap. Section 8 lists the full set of requirements.

In [ ]:
summary = report.summary({**results, "criteria": criteria})
(OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))